In [ ]:
"""
transformer_stock_ranking_predict_v21.py
V21 推理 — 加载 train_v21.py 输出的 JSON 权重, 预测测试区间.
架构/预处理与 train_v21.py 完全一致.
"""

import json, time, gc, base64
import numpy as np
import pandas as pd
import dai
import torch
import torch.nn as nn
import torch.nn.functional as F
import structlog

logger = structlog.get_logger()

SEQ_LEN = 240; SEED = 42; INFER_BATCH = 256; INFER_BATCH_INST = 200; DAI_CHUNK = 100
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RAW_FEATURES = ["open", "high", "low", "close",
                "bid_price1", "ask_price1",
                "volume", "amount", "bid_volume1", "ask_volume1"]
VOL_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]

# ═══════════ 模型定义 (与 train_v21.py 完全一致) ═══════════
class RotaryPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x, offset=0):
        seq_len = x.shape[1]
        t = torch.arange(seq_len, device=x.device) + offset
        freqs = torch.einsum("i,j->ij", t.float(), self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return torch.cos(emb).unsqueeze(0), torch.sin(emb).unsqueeze(0)

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos + rotate_half(q) * sin,
            k * cos + rotate_half(k) * sin)

class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, dim_ff, dropout=0.15):
        super().__init__()
        self.w1 = nn.Linear(d_model, dim_ff, bias=False)
        self.w2 = nn.Linear(d_model, dim_ff, bias=False)
        self.w3 = nn.Linear(dim_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, device=x.device)
        random_tensor.floor_()
        return x / keep_prob * random_tensor

class RoPETransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff, dropout=0.15, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn = SwiGLUFFN(d_model, dim_ff, dropout)
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()
        self.rope = RotaryPositionalEncoding(d_model // nhead)

    def forward(self, x, attn_mask=None):
        residual = x
        x_norm = self.norm1(x)
        q = k = v = x_norm
        cos, sin = self.rope(x_norm)
        B, L, D = q.shape
        H = self.attn.num_heads
        Dh = D // H
        q_r = q.view(B, L, H, Dh).transpose(1, 2)
        k_r = k.view(B, L, H, Dh).transpose(1, 2)
        v_r = v.view(B, L, H, Dh).transpose(1, 2)
        cos_r = cos.unsqueeze(1)
        sin_r = sin.unsqueeze(1)
        q_r, k_r = apply_rotary_pos_emb(q_r, k_r, cos_r, sin_r)
        q_h = q_r.transpose(1, 2).reshape(B, L, D)
        k_h = k_r.transpose(1, 2).reshape(B, L, D)
        v_h = v_r.transpose(1, 2).reshape(B, L, D)
        attn_out, _ = self.attn(q_h, k_h, v_h, need_weights=False, attn_mask=attn_mask)
        x = residual + self.drop_path(attn_out)
        residual = x
        x = residual + self.drop_path(self.ffn(self.norm2(x)))
        return x

class ConvTransformer(nn.Module):
    def __init__(self, n_feat=10, d_model=128, nhead=4, nlayers=4,
                 dim_ff=384, dropout=0.15, drop_path_rate=0.05):
        super().__init__()
        d1 = 42; d2 = 42; d3 = 44
        self.conv_s = nn.Sequential(
            nn.Conv1d(n_feat, d1, kernel_size=3, stride=2, padding=1), nn.GELU())
        self.conv_m = nn.Sequential(
            nn.Conv1d(n_feat, d2, kernel_size=5, stride=2, padding=2), nn.GELU())
        self.conv_l = nn.Sequential(
            nn.Conv1d(n_feat, d3, kernel_size=7, stride=2, padding=3), nn.GELU())
        self.conv_proj = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1), nn.GELU())
        self.seq_out = 60

        drop_paths = [drop_path_rate * (i / max(1, nlayers - 1))
                      for i in range(nlayers)]
        self.layers = nn.ModuleList([
            RoPETransformerLayer(d_model, nhead, dim_ff, dropout, drop_paths[i])
            for i in range(nlayers)
        ])
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 96), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(96, 1),
        )

    def forward(self, x):
        h_s = self.conv_s(x.permute(0, 2, 1))
        h_m = self.conv_m(x.permute(0, 2, 1))
        h_l = self.conv_l(x.permute(0, 2, 1))
        h = torch.cat([h_s, h_m, h_l], dim=1)
        h = self.conv_proj(h).permute(0, 2, 1)
        for layer in self.layers:
            h = layer(h)
        w = F.softmax(self.attn_pool(h), dim=1)
        h_pooled = (h * w).sum(dim=1)
        return self.head(h_pooled).squeeze(-1)

# ═══════════ 工具 ═══════════
@torch.no_grad()
def predict_batched(model, X_np, bs=INFER_BATCH):
    model.eval(); preds = []
    X_t = torch.from_numpy(X_np)
    for i in range(0, len(X_np), bs):
        preds.append(model(X_t[i:i+bs].to(device)).cpu().numpy())
    return np.concatenate(preds)

def build_features(df):
    feats = np.column_stack([df[c].to_numpy(np.float32) for c in RAW_FEATURES])
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    for ci, c in enumerate(RAW_FEATURES):
        if c in VOL_COLS:
            feats[:, ci] = np.log1p(feats[:, ci].clip(min=0))
    return feats.astype(np.float32)

def build_dataset(sd, ed, instruments, table, stats, chunk_size=DAI_CHUNK):
    t0 = time.time()
    buf = (pd.to_datetime(sd) - pd.Timedelta(days=90)).strftime("%Y-%m-%d")
    sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)
    wins, keys = [], []
    n_batches = (len(instruments) + chunk_size - 1) // chunk_size
    for ci in range(0, len(instruments), chunk_size):
        batch_inst = instruments[ci:ci+chunk_size]
        sql = f"SELECT date, instrument, {', '.join(RAW_FEATURES)} FROM {table} ORDER BY instrument, date"
        df = dai.query(sql, filters={"date": [buf, ed], "instrument": batch_inst}).df()
        df = df[df["date"].notna()]
        logger.info(f"  分批读取 {ci//chunk_size+1}/{n_batches} inst={len(batch_inst)} rows={len(df)}")
        for ins, sub in df.groupby("instrument", sort=False):
            if len(sub) <= SEQ_LEN: continue
            feats = build_features(sub)
            day = sub["date"].dt.normalize().to_numpy()
            close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))
            dates = day[close_pos]
            for k, p in enumerate(close_pos):
                d = pd.Timestamp(dates[k])
                if d < sd_ts or d > ed_ts: continue
                if p < SEQ_LEN: continue
                wins.append(feats[p - SEQ_LEN + 1:p + 1].copy())
                keys.append((d, ins))
        del df; gc.collect()
    if not keys: raise RuntimeError(f"无样本 {sd}~{ed}")
    X = np.stack(wins).astype(np.float32); del wins; gc.collect()
    m, s = stats
    X = (X - m[np.newaxis, np.newaxis, :]) / s[np.newaxis, np.newaxis, :]
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    idx_df = pd.DataFrame(keys, columns=["date", "instrument"])
    logger.info(f"数据 {sd[:7]}~{ed[:7]} n={len(keys)} t={round(time.time()-t0,1)}")
    return X, idx_df

def pool(sd, ed):
    return dai.query(
        "SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
        filters={"date": [sd, ed]}).df()["instrument"].tolist()

def load_model_stats(json_path):
    with open(json_path, "r") as f: p = json.load(f)
    model = ConvTransformer().to(device)
    flat = np.frombuffer(base64.b64decode(p["flat"]), dtype=np.dtype(p["dtype"]))
    state_dict = {}
    offset = 0
    for k, shape in zip(p["keys"], p["shapes"]):
        n = int(np.prod(shape))
        state_dict[k] = torch.from_numpy(flat[offset:offset+n].reshape(shape)).float()
        offset += n
    state_dict = {(k[7:] if k.startswith("module.") else k): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    mean = np.frombuffer(base64.b64decode(p["mean"]), dtype=np.dtype(p["stats_dtype"]))
    std  = np.frombuffer(base64.b64decode(p["std"]),  dtype=np.dtype(p["stats_dtype"]))
    stats = (mean.astype(np.float32), std.astype(np.float32))
    logger.info("模型已加载", path=json_path)
    return model, stats

def main(datasources, start_date, end_date):
    WEIGHT_FILE = "transformer_stock_ranking_v23_ep04.json"
    model, stats = load_model_stats(WEIGHT_FILE)

    test_table = datasources["bar1m"]
    logger.info("推理", start=str(start_date), end=str(end_date))
    all_inst = pool(start_date, end_date)
    results = []
    for b in range(0, len(all_inst), INFER_BATCH_INST):
        batch_inst = all_inst[b:b + INFER_BATCH_INST]
        Xte, bdf = build_dataset(start_date, end_date, batch_inst, table=test_table, stats=stats)
        bdf["score"] = predict_batched(model, Xte).astype(np.float64)
        results.append(bdf)
        del Xte; gc.collect()

    idx_df = pd.concat(results, ignore_index=True)
    del results; gc.collect()
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    result = (pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
              .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
              .drop_duplicates(["date", "instrument"])
              [["date", "instrument", "score"]].reset_index(drop=True))
    logger.info("输出", rows=len(result), days=result["date"].nunique())
    return result

if __name__ == "__main__":
    from bigmodule import M
    logger = structlog.get_logger()
    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数")
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
